In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install torchviz

In [ ]:
import os
import glob
import json
import copy
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
DATA_DIR = "/content/drive/MyDrive/processed_meg_regression"

BATCH_SIZE = 8
EPOCHS = 150
LR = 1e-4
WEIGHT_DECAY = 1e-4
RUN_SEED = 42

WINDOW_SIZE = 2048
WINDOW_STRIDE = 1024

EARLY_STOPPING_PATIENCE = 15
MIN_DELTA = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(RUN_SEED)
np.random.seed(RUN_SEED)
random.seed(RUN_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# JSON HELPER
# =========================================================
def to_python(obj):
    if isinstance(obj, dict):
        return {k: to_python(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_python(v) for v in obj]
    elif isinstance(obj, tuple):
        return [to_python(v) for v in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

# =========================================================
# WINDOWING
# =========================================================
def extract_windows_from_trial(trial_x, trial_y, window_size=2048, stride=1024):
    c, t = trial_x.shape
    xs = []
    start = 0

    while start + window_size <= t:
        xs.append(trial_x[:, start:start + window_size])
        start += stride

    if len(xs) == 0:
        return None, None

    return np.stack(xs).astype(np.float32), np.array(trial_y, dtype=np.float32)

# =========================================================
# LOADING
# =========================================================
def extract_subject_id_from_path(x_path):
    base = os.path.basename(x_path)
    return base.split("_")[0]

def load_all_trials_with_subjects(data_dir):
    x_paths = sorted(glob.glob(os.path.join(data_dir, "*_X.npy")))
    print(f"Searching in: {data_dir}")
    print(f"Found {len(x_paths)} X files")

    seq_x_list, y_list, subject_list = [], [], []

    for x_path in x_paths:
        y_path = x_path.replace("_X.npy", "_y.npy")
        if not os.path.exists(y_path):
            print(f"[WARN] Missing y file for: {x_path}")
            continue

        subject_id = extract_subject_id_from_path(x_path)
        X = np.load(x_path)
        y = np.load(y_path)

        n = min(len(X), len(y))
        X, y = X[:n], y[:n]

        for i in range(n):
            seq_x, y_trial = extract_windows_from_trial(
                X[i], y[i],
                window_size=WINDOW_SIZE,
                stride=WINDOW_STRIDE
            )
            if seq_x is not None:
                seq_x_list.append(seq_x)
                y_list.append(y_trial)
                subject_list.append(subject_id)

    print(f"Total usable trial sequences: {len(seq_x_list)}")
    print(f"Total unique subjects: {len(set(subject_list))}")
    return seq_x_list, y_list, subject_list

def filter_by_subjects(seq_x_list, y_list, subject_list, allowed_subjects):
    out_x, out_y = [], []
    for x, y, s in zip(seq_x_list, y_list, subject_list):
        if s in allowed_subjects:
            out_x.append(x)
            out_y.append(y)
    return out_x, out_y

# =========================================================
# DATASET
# =========================================================
class TrialDataset(Dataset):
    def __init__(self, seq_x_list, y_list):
        self.seq_x_list = seq_x_list
        self.y_list = y_list

    def __len__(self):
        return len(self.seq_x_list)

    def __getitem__(self, idx):
        x_seq = torch.tensor(self.seq_x_list[idx], dtype=torch.float32)
        y_trial = torch.tensor(self.y_list[idx], dtype=torch.float32)
        return x_seq, y_trial

# =========================================================
# NORMALIZATION
# =========================================================
def compute_train_stats(dataset_subset):
    xs = []
    ys = []

    for x_seq, y_trial in dataset_subset:
        xs.append(x_seq.numpy())
        ys.append(y_trial.numpy())

    X = np.concatenate(xs, axis=0)  # (all_windows, C, T)
    Y = np.stack(ys, axis=0)        # (N, 2)

    x_mean = X.mean(axis=(0, 2), keepdims=True)
    x_std = X.std(axis=(0, 2), keepdims=True) + 1e-6

    y_mean = Y.mean(axis=0, keepdims=True)
    y_std = Y.std(axis=0, keepdims=True) + 1e-6

    return (
        x_mean.astype(np.float32),
        x_std.astype(np.float32),
        y_mean.astype(np.float32),
        y_std.astype(np.float32),
    )

class NormalizedDataset(Dataset):
    def __init__(self, base_dataset, x_mean, x_std, y_mean, y_std):
        self.base_dataset = base_dataset
        self.x_mean = torch.tensor(x_mean, dtype=torch.float32)
        self.x_std = torch.tensor(x_std, dtype=torch.float32)
        self.y_mean = torch.tensor(y_mean, dtype=torch.float32)
        self.y_std = torch.tensor(y_std, dtype=torch.float32)

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        x_seq, y_trial = self.base_dataset[idx]

        x_seq = (x_seq - self.x_mean) / self.x_std
        x_seq = x_seq - x_seq.mean(dim=-1, keepdim=True)
        x_seq = x_seq / (x_seq.std(dim=-1, keepdim=True) + 1e-6)

        # extra amplitude stabilization for cross-subject robustness
        x_seq = x_seq / (x_seq.abs().max(dim=-1, keepdim=True)[0] + 1e-6)

        y_trial = (y_trial - self.y_mean.squeeze(0)) / self.y_std.squeeze(0)

        return x_seq, y_trial

# =========================================================
# MODEL
# =========================================================
class PlainCNNEncoder(nn.Module):
    def __init__(self, in_channels=143, feature_dim=256, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, feature_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.net(x)
        x = x.squeeze(-1)
        x = self.dropout(x)
        return x

class ImprovedLOSOBaseline(nn.Module):
    def __init__(self, in_channels=143, cnn_feature_dim=256, lstm_hidden=256, dropout=0.2):
        super().__init__()
        self.encoder = PlainCNNEncoder(
            in_channels=in_channels,
            feature_dim=cnn_feature_dim,
            dropout=dropout
        )

        self.temporal_bilstm = nn.LSTM(
            input_size=cnn_feature_dim,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.trial_head = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def forward(self, x_seq):
        B, W, C, T = x_seq.shape
        x = x_seq.reshape(B * W, C, T)
        z = self.encoder(x)
        z = z.reshape(B, W, -1)
        z_lstm, _ = self.temporal_bilstm(z)
        z_pool = z_lstm.mean(dim=1)
        pred_trial = self.trial_head(z_pool)
        return pred_trial

# =========================================================
# METRICS
# =========================================================
def rmse_numpy(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae_numpy(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def pearson_corr_numpy(y_true, y_pred):
    vals = []
    for i in range(y_true.shape[1]):
        a = y_true[:, i]
        b = y_pred[:, i]
        if np.std(a) < 1e-8 or np.std(b) < 1e-8:
            vals.append(0.0)
        else:
            vals.append(np.corrcoef(a, b)[0, 1])
    return vals

# =========================================================
# TRAIN / EVAL
# =========================================================
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x_seq, y_trial in loader:
        x_seq = x_seq.to(DEVICE)
        y_trial = y_trial.to(DEVICE)

        pred_trial = model(x_seq)
        loss = criterion(pred_trial, y_trial)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    return {"loss": total_loss / len(loader)}

@torch.no_grad()
def evaluate(model, loader, criterion, y_mean, y_std):
    model.eval()
    total_loss = 0.0
    all_true = []
    all_pred = []

    y_mean = y_mean.reshape(1, -1)
    y_std = y_std.reshape(1, -1)

    for x_seq, y_trial in loader:
        x_seq = x_seq.to(DEVICE)
        y_trial = y_trial.to(DEVICE)

        pred_trial = model(x_seq)
        loss = criterion(pred_trial, y_trial)
        total_loss += loss.item()

        y_true_denorm = y_trial.cpu().numpy() * y_std + y_mean
        y_pred_denorm = pred_trial.cpu().numpy() * y_std + y_mean

        all_true.append(y_true_denorm)
        all_pred.append(y_pred_denorm)

    y_true = np.concatenate(all_true, axis=0)
    y_pred = np.concatenate(all_pred, axis=0)

    rmse = rmse_numpy(y_true, y_pred)
    mae = mae_numpy(y_true, y_pred)
    r_v, r_a = pearson_corr_numpy(y_true, y_pred)

    return {
        "loss": total_loss / len(loader),
        "rmse": rmse,
        "mae": mae,
        "pearson_valence": r_v,
        "pearson_arousal": r_a,
        "y_true": y_true,
        "y_pred": y_pred,
    }

# =========================================================
# CURVE PLOTTING
# =========================================================
def save_fold_curves(history, held_out_subject):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Loss Curves - Held-out {held_out_subject}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"curve_loss_loso_{held_out_subject}.png")
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["r_v"], label="Val r(V)")
    plt.plot(epochs, history["r_a"], label="Val r(A)")
    plt.plot(epochs, history["score"], label="Val Score")
    plt.xlabel("Epoch")
    plt.ylabel("Correlation / Score")
    plt.title(f"Validation Curves - Held-out {held_out_subject}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"curve_metrics_loso_{held_out_subject}.png")
    plt.close()

# =========================================================
# TRAIN ONE LOSO FOLD
# =========================================================
def train_one_loso_fold(held_out_subject, seq_x_list, y_list, subject_list):
    train_subjects = sorted(list(set(subject_list) - {held_out_subject}))
    val_subjects = [held_out_subject]

    train_x, train_y = filter_by_subjects(seq_x_list, y_list, subject_list, set(train_subjects))
    val_x, val_y = filter_by_subjects(seq_x_list, y_list, subject_list, set(val_subjects))

    print("\n" + "=" * 70)
    print(f"LOSO FOLD | Held-out subject: {held_out_subject}")
    print("=" * 70)
    print(f"Train trials: {len(train_x)}")
    print(f"Val trials:   {len(val_x)}")

    train_base = TrialDataset(train_x, train_y)
    val_base = TrialDataset(val_x, val_y)

    x_mean, x_std, y_mean, y_std = compute_train_stats(train_base)
    train_set = NormalizedDataset(train_base, x_mean, x_std, y_mean, y_std)
    val_set = NormalizedDataset(val_base, x_mean, x_std, y_mean, y_std)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model = ImprovedLOSOBaseline(
        in_channels=143,
        cnn_feature_dim=256,
        lstm_hidden=256,
        dropout=0.2
    ).to(DEVICE)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=10
    )

    best_score = -float("inf")
    best_epoch = -1
    best_checkpoint = None
    epochs_without_improvement = 0

    fold_history = {
        "train_loss": [],
        "val_loss": [],
        "rmse": [],
        "mae": [],
        "r_v": [],
        "r_a": [],
        "score": []
    }

    for epoch in range(1, EPOCHS + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion)
        val_metrics = evaluate(model, val_loader, criterion, y_mean, y_std)

        score = 0.5 * (val_metrics["pearson_valence"] + val_metrics["pearson_arousal"])
        scheduler.step(score)

        fold_history["train_loss"].append(train_metrics["loss"])
        fold_history["val_loss"].append(val_metrics["loss"])
        fold_history["rmse"].append(val_metrics["rmse"])
        fold_history["mae"].append(val_metrics["mae"])
        fold_history["r_v"].append(val_metrics["pearson_valence"])
        fold_history["r_a"].append(val_metrics["pearson_arousal"])
        fold_history["score"].append(score)

        print(
            f"Held-out {held_out_subject} | Epoch {epoch:02d} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE: {val_metrics['rmse']:.4f} | "
            f"Val MAE: {val_metrics['mae']:.4f} | "
            f"Val r(V): {val_metrics['pearson_valence']:.4f} | "
            f"Val r(A): {val_metrics['pearson_arousal']:.4f} | "
            f"Score: {score:.4f}"
        )

        improved = score > (best_score + MIN_DELTA)

        if improved:
            best_score = score
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "held_out_subject": held_out_subject,
                "epoch": epoch,
                "score": score,
                "valence_r": val_metrics["pearson_valence"],
                "arousal_r": val_metrics["pearson_arousal"],
                "rmse": val_metrics["rmse"],
                "mae": val_metrics["mae"],
                "y_true": val_metrics["y_true"],
                "y_pred": val_metrics["y_pred"],
            }

            torch.save(
                {
                    "held_out_subject": held_out_subject,
                    "epoch": epoch,
                    "model_state_dict": copy.deepcopy(model.state_dict()),
                    "score": score,
                    "valence_r": val_metrics["pearson_valence"],
                    "arousal_r": val_metrics["pearson_arousal"],
                    "rmse": val_metrics["rmse"],
                    "mae": val_metrics["mae"],
                },
                f"best_model_loso_improved_{held_out_subject}.pt"
            )

            print(
                f"🔥 Best improved fold updated | "
                f"Epoch {epoch} | "
                f"r(V): {val_metrics['pearson_valence']:.4f} | "
                f"r(A): {val_metrics['pearson_arousal']:.4f} | "
                f"Score: {score:.4f}"
            )
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping for held-out {held_out_subject} at epoch {epoch}. "
                f"Best epoch was {best_epoch}."
            )
            break

    np.save(f"best_y_true_loso_improved_{held_out_subject}.npy", best_checkpoint["y_true"])
    np.save(f"best_y_pred_loso_improved_{held_out_subject}.npy", best_checkpoint["y_pred"])

    save_fold_curves(fold_history, held_out_subject)

    with open(f"history_loso_improved_{held_out_subject}.json", "w") as f:
        json.dump(to_python(fold_history), f, indent=4)

    return {
        "held_out_subject": held_out_subject,
        "best_epoch": best_checkpoint["epoch"],
        "score": best_checkpoint["score"],
        "pearson_valence": best_checkpoint["valence_r"],
        "pearson_arousal": best_checkpoint["arousal_r"],
        "rmse": best_checkpoint["rmse"],
        "mae": best_checkpoint["mae"],
    }

# =========================================================
# MAIN
# =========================================================
def main():
    print("Loading processed trials...")
    seq_x_list, y_list, subject_list = load_all_trials_with_subjects(DATA_DIR)

    if len(seq_x_list) == 0:
        raise FileNotFoundError(f"No processed trials found in {DATA_DIR}.")

    print("Example x_seq shape:", seq_x_list[0].shape)
    print("Example y shape:", y_list[0].shape)

    unique_subjects = sorted(list(set(subject_list)))
    print(f"\nRunning improved LOSO baseline across {len(unique_subjects)} subjects:")
    print(unique_subjects)

    config = {
        "model": "ImprovedLOSOBaseline",
        "cnn_feature_dim": 256,
        "lstm_hidden": 256,
        "dropout": 0.2,
        "lr": LR,
        "epochs": EPOCHS,
        "evaluation": "LOSO",
        "subjects": unique_subjects,
    }

    with open("loso_improved_baseline_config.json", "w") as f:
        json.dump(to_python(config), f, indent=4)

    all_fold_results = []

    for held_out_subject in unique_subjects:
        fold_result = train_one_loso_fold(
            held_out_subject,
            seq_x_list,
            y_list,
            subject_list
        )
        all_fold_results.append(fold_result)

    print("\n" + "=" * 70)
    print("IMPROVED LOSO PER-SUBJECT RESULTS")
    print("=" * 70)
    for r in all_fold_results:
        print(
            f"Held-out {r['held_out_subject']} | "
            f"Best Epoch: {r['best_epoch']} | "
            f"Score: {r['score']:.4f} | "
            f"r(V): {r['pearson_valence']:.4f} | "
            f"r(A): {r['pearson_arousal']:.4f} | "
            f"RMSE: {r['rmse']:.4f} | "
            f"MAE: {r['mae']:.4f}"
        )

    rv = np.array([r["pearson_valence"] for r in all_fold_results], dtype=np.float32)
    ra = np.array([r["pearson_arousal"] for r in all_fold_results], dtype=np.float32)
    rmse = np.array([r["rmse"] for r in all_fold_results], dtype=np.float32)
    mae = np.array([r["mae"] for r in all_fold_results], dtype=np.float32)
    score = np.array([r["score"] for r in all_fold_results], dtype=np.float32)

    summary = {
        "rV_mean": float(rv.mean()),
        "rV_std": float(rv.std()),
        "rA_mean": float(ra.mean()),
        "rA_std": float(ra.std()),
        "rmse_mean": float(rmse.mean()),
        "rmse_std": float(rmse.std()),
        "mae_mean": float(mae.mean()),
        "mae_std": float(mae.std()),
        "score_mean": float(score.mean()),
        "score_std": float(score.std()),
        "per_subject": all_fold_results,
    }

    with open("loso_improved_baseline_summary.json", "w") as f:
        json.dump(to_python(summary), f, indent=4)

    print("\n" + "=" * 70)
    print("IMPROVED LOSO FINAL SUMMARY")
    print("=" * 70)
    print(f"Valence r (rV): {summary['rV_mean']:.4f} ± {summary['rV_std']:.4f}")
    print(f"Arousal r (rA): {summary['rA_mean']:.4f} ± {summary['rA_std']:.4f}")
    print(f"RMSE:           {summary['rmse_mean']:.4f} ± {summary['rmse_std']:.4f}")
    print(f"MAE:            {summary['mae_mean']:.4f} ± {summary['mae_std']:.4f}")
    print(f"Score:          {summary['score_mean']:.4f} ± {summary['score_std']:.4f}")

    print("\nSaved:")
    print("  loso_improved_baseline_config.json")
    print("  loso_improved_baseline_summary.json")
    print("  best_model_loso_improved_<subject>.pt")
    print("  best_y_true_loso_improved_<subject>.npy")
    print("  best_y_pred_loso_improved_<subject>.npy")
    print("  history_loso_improved_<subject>.json")
    print("  curve_loss_loso_<subject>.png")
    print("  curve_metrics_loso_<subject>.png")


if __name__ == "__main__":
    main()

Loading processed trials...
Searching in: /content/drive/MyDrive/processed_meg_regression
Found 82 X files
Total usable trial sequences: 814
Total unique subjects: 21
Example x_seq shape: (4, 143, 2048)
Example y shape: (2,)

Running improved LOSO baseline across 21 subjects:
['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18', 'sub-19', 'sub-20', 'sub-21', 'sub-22', 'sub-23']

LOSO FOLD | Held-out subject: sub-01
Train trials: 774
Val trials:   40
Held-out sub-01 | Epoch 01 | Train Loss: 0.9712 | Val Loss: 0.7915 | Val RMSE: 0.2797 | Val MAE: 0.2459 | Val r(V): -0.2249 | Val r(A): 0.3544 | Score: 0.0648
🔥 Best improved fold updated | Epoch 1 | r(V): -0.2249 | r(A): 0.3544 | Score: 0.0648
Held-out sub-01 | Epoch 02 | Train Loss: 0.8768 | Val Loss: 0.9370 | Val RMSE: 0.3008 | Val MAE: 0.2531 | Val r(V): 0.3889 | Val r(A): -0.0510 | Score: 0.1690
🔥 Best improved fold updated | Epoc

### Saved Files Overview

During the training process, several files were saved for each held-out subject:

*   `best_model_loso_improved_<subject>.pt`: The PyTorch model checkpoint from the best epoch for that subject.
*   `best_y_true_loso_improved_<subject>.npy`: The true target values for the validation set of that subject.
*   `best_y_pred_loso_improved_<subject>.npy`: The predicted target values for the validation set of that subject.
*   `history_loso_improved_<subject>.json`: A JSON file containing the training history (loss, metrics) for the fold.
*   `curve_loss_loso_<subject>.png`: A plot showing the train and validation loss curves.
*   `curve_metrics_loso_<subject>.png`: A plot showing validation correlation metrics curves.

In [ ]:
import glob
from google.colab import files

# List all generated files
generated_files = glob.glob('*.json') + glob.glob('*.pt') + glob.glob('*.npy') + glob.glob('*.png')
print("Files generated during training:")
for f in generated_files:
    print(f)

# You can download all files with the following code (uncomment to run):
for f in generated_files:
    files.download(f)

Files generated during training:
history_loso_improved_sub-17.json
history_loso_improved_sub-01.json
loso_improved_baseline_summary.json
history_loso_improved_sub-15.json
history_loso_improved_sub-06.json
history_loso_improved_sub-18.json
history_loso_improved_sub-02.json
history_loso_improved_sub-10.json
history_loso_improved_sub-23.json
history_loso_improved_sub-03.json
history_loso_improved_sub-20.json
history_loso_improved_sub-21.json
history_loso_improved_sub-04.json
history_loso_improved_sub-14.json
history_loso_improved_sub-19.json
history_loso_improved_sub-07.json
history_loso_improved_sub-22.json
history_loso_improved_sub-05.json
history_loso_improved_sub-09.json
history_loso_improved_sub-08.json
history_loso_improved_sub-11.json
history_loso_improved_sub-16.json
loso_improved_baseline_config.json
best_model_loso_improved_sub-09.pt
best_model_loso_improved_sub-16.pt
best_model_loso_improved_sub-15.pt
best_model_loso_improved_sub-17.pt
best_model_loso_improved_sub-07.pt
best_mo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>